## Topic 10: The Honest Migration Trigger — When In-Memory Stops Being Enough

### Concept, Intuition, Why It Exists

- Every earlier topic in this chapter presented Qdrant as a clear improvement over the in-memory store. This topic deliberately asks the opposite question: when is the in-memory store actually still fine, and what are the real, concrete signals that tell you it has stopped being fine?
- This matters because premature migration is a real cost — Docker setup, collection management, persistence infrastructure, monitoring — and paying that cost before it's justified adds operational burden with no user-visible benefit. The goal is to be able to say, with precision, "we haven't hit any of these triggers yet, so migration would be premature" or "trigger X happened on this date, which is why we migrated."
- There is no universal answer. The triggers are context-dependent: a process that restarts once a week and takes 30 seconds to re-embed is a different situation from a process that restarts multiple times a day and takes 20 minutes to re-embed. What follows is a framework for making the decision explicitly rather than migrating out of vague anxiety or staying in-memory out of laziness.

### The Three Honest Triggers

**Trigger 1 — Re-embedding on restart has become costly**

- The in-memory store re-embeds the entire corpus every time the process starts. At 17 JSON pages and a handful of CSV rows, this takes a few seconds. At a real corpus of thousands of chunks, it can take minutes.
- The trigger is not "the corpus is large" in abstract — it's "restart now takes long enough to be a real operational problem." Concretely: if an on-call engineer restarts the service during an incident and has to wait several minutes before the service is ready, that is the trigger. If restart takes under 30 seconds, it probably isn't.
- The honest question is: what is the actual measured restart time today, and is that number growing as the corpus grows?

**Trigger 2 — Metadata filtering after retrieval is returning fewer than k useful results**

- The in-memory store's filtered search fetches all results and discards non-matching ones afterward. If most chunks are policy documents and a search for FAQ content asks for k=5, it might return only 2 useful results after discarding.
- The trigger is not "we need filtering" in abstract — it's "post-search filtering is measurably returning fewer useful results than the requested k, often enough to affect answer quality." If filtering is rare and the corpus has a roughly even distribution of types, post-search filtering may be fine for a long time.
- The honest question is: for the actual query patterns in production, how often does post-search filtering return fewer than k useful results?

**Trigger 3 — Brute-force scan latency is measurably affecting query response time**

- The in-memory store scans every vector on every query. At a few hundred chunks, each scan is microseconds. At tens of thousands of chunks, it becomes milliseconds that add up across a query pipeline that already includes an LLM call.
- The trigger is not "the corpus has more than N chunks" — it's "measured query latency has grown to the point where the vector search step is a meaningful fraction of total query time, and that fraction is growing." A pipeline where the LLM call takes 2 seconds and vector search takes 5 milliseconds is fine. The same pipeline where vector search takes 500 milliseconds is not.
- The honest question is: what is the actual measured vector search time today, and what will it be when the corpus is 10x its current size?

### A Fourth Trigger Specific to This Project

**Trigger 4 — The PII concern becomes real**

- Topic 9 covered why customer records in a vector store require caller-scoped filtering, right-to-be-forgotten support, and architectural separation from policy documents. The in-memory store has none of these — it's a flat list with no identity awareness and no deletion mechanism.
- The trigger is not "we have customer records in the store" — it's "we have a compliance requirement, an audit, or a live product that makes the in-memory store's lack of access control a real risk rather than a theoretical one."
- A prototype that only runs locally and never serves real customer data can stay in-memory without PII concern. A production system that serves a bank's customers cannot.

### How It's Implemented In This Project

- This project currently sits comfortably before all four triggers: the corpus is 17 JSON pages plus a handful of CSV rows, restart takes seconds, all query patterns are exploratory rather than production-facing, and the system does not serve real customer data. The in-memory store with `QdrantClient(":memory:")` is genuinely the right choice at this stage.
- The Qdrant client code is already written using `:memory:` mode, so when trigger 1, 2, 3, or 4 actually fires, the migration is a one-line change to `get_client()` plus the Docker setup from Topic 3. The code cost of migration is deliberately near-zero because the application code never needed to know which mode was active.

### Real-World Issues, Edge Cases, Debugging

- **The triggers are lagging indicators if you're not measuring**: re-embedding time, query latency, and filtering effectiveness are invisible unless someone is actively tracking them. A system can cross all three triggers gradually over months with no single moment where it becomes obviously broken — it just slowly gets worse. Setting up basic measurements before they become urgent is the difference between a planned migration and an emergency one.
- **Corpus size is a proxy, not the trigger**: "we now have 10,000 chunks so we should migrate" is a guess. "Re-embedding takes 8 minutes and that is blocking incident response" is a trigger. Always trace back to the actual measured impact, not a round number.
- **Migration doesn't have to be all-or-nothing**: it's valid to migrate the customer records collection to persistent Qdrant (trigger 4) while keeping policy documents in-memory (triggers 1, 2, 3 not yet hit). The collections are independent — each can be migrated on its own timeline when its specific trigger fires.

### Design Decisions & Trade-offs

- Staying in-memory longer than necessary is a real cost: re-embedding overhead on every restart, risk of data loss on unexpected crashes, no filtering guarantees. These are real but often small costs at early stage.
- Migrating earlier than necessary is also a real cost: Docker infrastructure, collection management, persistence monitoring, backup strategy, handling the Docker volume correctly, debugging the occasional container restart. These are also real costs that compound with team size and operational maturity.
- The right answer is explicitly tracking the actual measured values for all four triggers and migrating when a trigger actually fires, not before and not much after.

### Alternatives & When To Use Each

- Stay in-memory indefinitely — appropriate only for scripts, notebooks, CI tests, and one-off analyses where none of the four triggers can fire by definition.
- Migrate to `:memory:` Qdrant (already done) — gains filtering and the correct data model at zero infrastructure cost, but doesn't solve persistence or PII concerns.
- Migrate to local Docker Qdrant — solves triggers 1 and 3 (persistence eliminates re-embedding cost, HNSW eliminates scan latency). Trigger 2 is solved at any Qdrant mode. Trigger 4 depends on network-level access control around the Docker container, not just the client.
- Migrate to Qdrant Cloud — solves all four triggers plus adds managed persistence, backups, and network-level access control. Appropriate when Docker management itself becomes an operational burden.

### Common Mistakes & Production Failures

- Migrating preemptively to a more complex system before any trigger fires, then spending operational time managing infrastructure that adds no user-visible value at current scale.
- Staying in-memory past the point where trigger 1 has clearly fired, then discovering during an incident that a service restart takes 15 minutes and the on-call engineer has no good options.
- Treating corpus size as the migration trigger rather than the actual measured impact — a 50,000-chunk corpus with fast embeddings and rare restarts might not need migration; a 500-chunk corpus with slow embeddings and frequent restarts might.
- Not measuring any of the four trigger signals in production, and therefore not knowing when any of them have fired until someone reports a problem.

### Lead-Level Interview Questions

**Q: How would you make the case to a team that "we don't need Qdrant yet" without it sounding like avoiding work?**
A: Frame it around the four concrete triggers and show they haven't fired: re-embedding time is X seconds (acceptable), query latency from vector search is Y milliseconds (negligible fraction of total), post-search filtering is returning k results reliably (no undercount), and we have no live customer data in the store (PII concern is theoretical). Migration would add Z operational concerns for no improvement the user would see. We revisit when one of those four measurements changes. That is a specific, measurable statement, not a general "we don't need it yet."

**Q: At what corpus size would you migrate from in-memory to Qdrant with Docker?**
A: I don't have a corpus size answer because that's not the right framing. The triggers are: restart time, query latency, filtering effectiveness, and PII compliance requirements. Those are what determine whether migration is justified. A corpus of 100,000 tiny chunks might have a fast enough embedding model that restart is still acceptable. A corpus of 5,000 large chunks with a slow model might have crossed trigger 1 already. Measuring the actual values is the only honest way to answer the question.

**Q: The team says "let's just use Qdrant from the start to avoid having to migrate later." How do you evaluate that?**
A: It's a reasonable position if the team already has Docker experience, the operational overhead is genuinely low, and the project is likely to reach migration-trigger scale quickly. It becomes unreasonable if it means spending days on infrastructure setup before the core product has been validated, or if it adds complexity that slows down iteration during the period when fast iteration matters most. The right answer depends on the team's existing operational maturity and the project's stage — not on an abstract preference for one architecture over another.

**Q: What is the one-line change that migrates this project from in-memory to persistent Docker Qdrant?**
A: Change `QdrantClient(":memory:")` to `QdrantClient(url="http://localhost:6333")` in `get_client()`, after running the Docker container with a volume mount. Every other line of code — collection creation, upsert, search, filtering — stays identical. This is the payoff of having written all application code against the Qdrant client abstraction from the start, regardless of which mode it was targeting.

### Hidden Concepts Worth Knowing

- **Operational maturity as a hidden factor**: the right migration timing also depends on the team's ability to operate the new infrastructure reliably. A team that has never run a Docker container in production and doesn't have monitoring or backup procedures isn't ready to benefit from persistent Qdrant even if trigger 1 has technically fired — the infrastructure cost outweighs the benefit until the operational maturity catches up.
- **The sunk cost trap**: once a team has invested time in Qdrant setup, there's a natural tendency to keep using it even if the project's direction changes and the triggers no longer apply. Infrastructure decisions should be revisited when use cases change, not preserved because they were correct once.

### Revision Summary

> The in-memory store is correct and the right starting point. Migration is justified when one of four concrete triggers actually fires: restart re-embedding has become costly enough to affect operations, post-search filtering returns fewer than k useful results often enough to affect quality, vector scan latency is a measurable fraction of query time, or PII compliance requires access control the in-memory store cannot provide. Trigger points are measured values, not corpus size thresholds. The migration itself is a one-line change to `get_client()` — the application code never needed to know which mode was active.

In [ ]:
"""
migration_trigger_checker.py
------------------------------
A minimal measurement harness that tracks the four migration triggers.
Run this periodically as the corpus grows to know when migration is
actually justified rather than guessing from corpus size alone.

Uses :memory: mode to simulate the in-memory store being measured.
Install: pip install qdrant-client sentence-transformers
"""

import time
import hashlib
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct, Filter, FieldCondition, MatchValue
from sentence_transformers import SentenceTransformer

MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"
VECTOR_SIZE = 384


def measure_trigger_1_restart_cost(chunks: list, model: SentenceTransformer) -> float:
    """Trigger 1: how long does a full re-embed take on restart?
    If this number crosses ~30 seconds, migration is worth considering."""
    start = time.time()

    # simulate restart: re-embed everything from scratch
    texts = [c["text"] for c in chunks]
    model.encode(texts, normalize_embeddings=True)

    elapsed = time.time() - start
    print(f"Trigger 1 — Restart re-embed time: {elapsed:.2f}s for {len(chunks)} chunks")
    print(f"  {'OK' if elapsed < 30 else 'TRIGGER FIRED'} (threshold: 30s)")
    return elapsed


def measure_trigger_2_filter_effectiveness(
    client: QdrantClient, collection: str, model: SentenceTransformer,
    query: str, target_doc_type: str, k: int = 5
) -> int:
    """Trigger 2: does post-search filtering return fewer than k useful results?
    Simulates the in-memory approach: fetch all, filter afterward."""
    query_vector = model.encode(query, normalize_embeddings=True).tolist()

    # fetch more than k to simulate post-search filtering
    all_results = client.query_points(
        collection_name=collection,
        query=query_vector,
        limit=50,  # over-fetch
        with_payload=True,
    ).points

    # filter afterward -- the in-memory store's approach
    filtered = [r for r in all_results if r.payload.get("doc_type") == target_doc_type]
    useful_count = len(filtered[:k])

    print(f"Trigger 2 — Post-filter useful results: {useful_count}/{k} "
          f"(doc_type='{target_doc_type}')")
    print(f"  {'OK' if useful_count >= k else 'TRIGGER FIRED'} (threshold: {k} useful results)")
    return useful_count


def measure_trigger_3_search_latency(
    client: QdrantClient, collection: str, model: SentenceTransformer,
    query: str, n_runs: int = 10
) -> float:
    """Trigger 3: what is the average vector search latency?
    If this is a meaningful fraction of total query time, migration helps."""
    query_vector = model.encode(query, normalize_embeddings=True).tolist()

    times = []
    for _ in range(n_runs):
        start = time.time()
        client.query_points(
            collection_name=collection,
            query=query_vector,
            limit=5,
            with_payload=False,
        )
        times.append(time.time() - start)

    avg_ms = (sum(times) / len(times)) * 1000
    print(f"Trigger 3 — Average search latency: {avg_ms:.1f}ms over {n_runs} runs "
          f"({client.get_collection(collection).points_count} points)")
    print(f"  {'OK' if avg_ms < 100 else 'TRIGGER FIRED'} (threshold: 100ms)")
    return avg_ms


def measure_trigger_4_pii_in_store(client: QdrantClient, collection: str) -> bool:
    """Trigger 4: does the store contain PII fields?
    If yes and the system serves real users, migration to scoped access is needed."""
    results, _ = client.scroll(
        collection_name=collection,
        limit=1,
        with_payload=True,
        with_vectors=False,
    )
    if not results:
        print("Trigger 4 — No points in collection")
        return False

    pii_fields = {"customer_name", "mobile", "account_no", "mobile_number"}
    payload_keys = set(results[0].payload.keys())
    has_pii = bool(pii_fields & payload_keys)

    print(f"Trigger 4 — PII fields in payload: {pii_fields & payload_keys or 'none'}")
    print(f"  {'TRIGGER FIRED (if serving real users)' if has_pii else 'OK'}")
    return has_pii


if __name__ == "__main__":
    model = SentenceTransformer(MODEL_NAME)
    client = QdrantClient(":memory:")

    # build a small test collection
    CHUNKS = [
        {"text": "Premature withdrawal incurs a 1 percent penalty.", "doc_type": "policy"},
        {"text": "Senior citizens get 0.5 percent extra interest.", "doc_type": "product"},
        {"text": "What is the minimum FD amount? Rs 25,000.", "doc_type": "faq"},
        {"text": "The FD interest rate for 24 months is 7.1 percent.", "doc_type": "product"},
        {"text": "Nomination facility is available for all FD accounts.", "doc_type": "policy"},
    ]
    client.create_collection("test", vectors_config=VectorParams(size=VECTOR_SIZE, distance=Distance.COSINE))
    texts = [c["text"] for c in CHUNKS]
    embs = model.encode(texts, normalize_embeddings=True)
    client.upsert("test", points=[
        PointStruct(
            id=int(hashlib.md5(CHUNKS[i]["text"].encode()).hexdigest()[:8], 16),
            vector=embs[i].tolist(),
            payload=CHUNKS[i],
        )
        for i in range(len(CHUNKS))
    ])

    print("=" * 55)
    print("MIGRATION TRIGGER MEASUREMENTS")
    print("=" * 55)
    measure_trigger_1_restart_cost(CHUNKS, model)
    print()
    measure_trigger_2_filter_effectiveness(client, "test", model,
                                           "penalty for early closure", "faq", k=3)
    print()
    measure_trigger_3_search_latency(client, "test", model,
                                     "premature withdrawal")
    print()
    measure_trigger_4_pii_in_store(client, "test")
    print()
    print("=" * 55)
    print("Run this periodically as corpus grows.")
    print("Migrate when a trigger fires, not before.")
    print("=" * 55)